Notebook: feature_engineering.ipynb  
Project: EcoPackAI  

This notebook creates engineered sustainability and performance features:
- CO₂ Impact Index (CII)
- Cost Efficiency Index (CEI)
- Material Suitability Score (MSS)


In [42]:
# cell 1 Environment setup
import pandas as pd
import numpy as np


In [43]:
#cell 2 Data loading
material = pd.read_csv(r'C:\Users\Sneha\OneDrive\Desktop\EcoPackAI\ml\data\processed\material_cleaned.csv')
print("material and product data loaded successfully")
print(material.shape)
material.head()


material and product data loaded successfully
(140, 9)


,material_id,material_type,strength_mpa,weight_capacity,biodegradability_percent,co2_emission_score,recyclability_percent,cost_per_kg,industry_use_case
0,1,Recycled Paper,45.7,1.58,96.1,26.9,69.8,44.70,Household
1,2,Glassine,73.4,11.53,22.1,38.4,81.8,186.06,Pharmacy
2,3,Compostable Film,68.1,7.96,79.8,8.9,76.3,111.77,Electronics
3,4,Paper,43.0,6.60,63.3,9.0,94.1,42.29,Pharmacy
4,5,Hemp Fiber,32.8,0.87,81.9,12.6,50.2,33.84,Apparel


In [44]:
#cell 3 checking missing values
material.isnull().sum()

material_id                 0
material_type               0
strength_mpa                0
weight_capacity             0
biodegradability_percent    0
co2_emission_score          0
recyclability_percent       0
cost_per_kg                 0
industry_use_case           0
dtype: int64

In [45]:
# cell 4- Safe Min-Max Normalization
def min_max_safe(series):
    if series.max() == series.min():
        return np.zeros(len(series))
    return (series - series.min()) / (series.max() - series.min())


In [46]:
# cell 5 — CO₂ Impact Index 
co2_norm = 1 - min_max_safe(material["co2_emission_score"])
bio_norm = min_max_safe(material["biodegradability_percent"])
recycle_norm = min_max_safe(material["recyclability_percent"])

material["co2_impact_index"] = (
    0.4 * co2_norm +
    0.3 * bio_norm +
    0.3 * recycle_norm
) * 100

material["co2_impact_index"] = material["co2_impact_index"].clip(0, 100)




In [47]:
# cell 6 Cost Efficiency Index
cost_norm = 1 - min_max_safe(material["cost_per_kg"])
weight_norm = min_max_safe(material["weight_capacity"])
strength_norm = min_max_safe(material["strength_mpa"])

material["cost_efficiency_index"] = (
    0.4 * cost_norm +
    0.3 * weight_norm +
    0.3 * strength_norm
) * 100

material["cost_efficiency_index"] = material["cost_efficiency_index"].clip(0, 100)




In [51]:
#cell 7 Material Suitability Score
strength_norm_mss = min_max_safe(material["strength_mpa"])
weight_norm_mss = min_max_safe(material["weight_capacity"])

# Base numeric suitability score
base_mss = (
    0.6 * strength_norm_mss +
    0.4 * weight_norm_mss
)

# Industry boost (normalized influence)
industry_boost = {
    "Pharmacy": 1.0,
    "Electronics": 0.8,
    "Household": 0.5,
    "Apparel": 0.4
}

industry_factor = material["industry_use_case"].map(industry_boost).fillna(0.3)

# Final MSS
material["material_suitability_score"] = (
    0.85 * base_mss +
    0.15 * industry_factor
) * 100

material["material_suitability_score"] = material["material_suitability_score"].clip(0, 100)



In [49]:
#cell 8
material[
    ["co2_impact_index",
     "cost_efficiency_index",
     "material_suitability_score"]
].describe()



,co2_impact_index,cost_efficiency_index,material_suitability_score
count,140.000000,140.000000,140.000000
mean,77.338561,49.110216,39.936652
std,16.110511,10.556288,17.860559
min,2.383178,5.019617,5.018579
25%,74.876919,43.179302,26.985972
50%,80.303086,49.992624,38.793708
75%,86.731086,55.498192,48.675873
max,94.859149,76.209460,94.009580


In [53]:
#cell 9 Save engineered features
material.to_csv("../data/processed/material_engineered.csv", index=False)
